# Heart Disease Risk Classifier — step-by-step mirror

This notebook runs the same pipeline as the `.py` modules (data_prep.py, features.py, train.py, explain.py) and reads the already-trained artifacts, so you can inspect each step without re-training.

> Demonstration prototype only — not medical advice.

In [ ]:
# 1. Data prep
import data_prep

raw = data_prep.load_heart_data(data_dir='data')
cleaned = data_prep.clean(raw)
print('Missing values:')
print(data_prep.missing_summary(cleaned.df).to_string())
print('\nClass balance:', cleaned.target.value_counts().to_dict())
print('\nColumn -> report group:')
print(cleaned.report_map.to_string(index=False))

In [ ]:
# 2. Feature engineering
import features

X_clean = data_prep.impute_median(cleaned.df)
X = features.build_features(X_clean)
print('Engineered features added:', features.ADDED_FEATURES)
print(X[features.ADDED_FEATURES].describe().round(3).to_string())

In [ ]:
# 3. Model metrics (read from the saved artifact; trained by train.py)
import json, pickle

metrics = json.load(open('artifacts/model_metrics.json'))
for name, v in metrics['cv_summary'].items():
    print(f"{name:28s} accuracy={v['accuracy']:.3f} recall={v['recall']:.3f}")
print('\nSelected model:', metrics['selected_model'], '| best RF params:', metrics['best_rf_params'])
print('Risk bands:', metrics['risk_bands'])
print('Held-out test:', metrics['test'])

In [ ]:
# 4. SHAP explanation for a single patient
import explain, pandas as pd

art = explain.load_artifact('artifacts/model.pkl')
row = {c: 0 for c in art['input_columns']}
row.update({'age':63,'sex':1,'cp':4,'trestbps':145,'chol':233,'fbs':1,'restecg':0,'thalach':150,'exang':1,'oldpeak':2.3,'slope':2,'ca':0,'thal':7})
df = pd.DataFrame([row])[art['input_columns']]
prob = art['model'].predict_proba(df)[:,1][0]
print('Disease probability:', round(prob,3))
res = explain.explain_patient(art, dict(df.iloc[0]), n_top=5)
print('In words:', explain.summarize_in_words(res))
for f in res['top_features']:
    print(f"  {f['label']:42s} shap={f['shap']:+.4f} ({f['direction']})")

## Honest numbers

90% accuracy was **not** reached on this small dataset (~303 rows, overlapping classes). 5-fold CV: baseline LR ~84.3% / tuned RF ~83.1% accuracy, ~78-79% recall. Lowering the decision threshold to hit ≥90% recall lifts test recall to ~100% but drops test accuracy to ~77% (false positives). See README.md for the full discussion.